# Домашнее задание 4.3: Бассейны Ньютона при слиянии корней (🟢)

Семейство полиномов

$$f_\varepsilon(z) = (z+1)(z-1)(z-\varepsilon), \qquad z \in \mathbb{C}.$$

При $\varepsilon \to 1$ два корня сближаются, при $\varepsilon = 1$ сливаются в двойной. На
зелёном уровне беру $\varepsilon = 0$, то есть $f_0(z) = z^3 - z$ с корнями $-1, 0, 1$, и
строю бассейны для итерации Ньютона

$$z_{n+1} = z_n - \frac{f_\varepsilon(z_n)}{f_\varepsilon'(z_n)}.$$

Область $\mathrm{Re}\,z, \mathrm{Im}\,z \in [-2, 2]$, сетка $600\times600$, не больше 100
итераций. Код написан сразу под произвольные $\varepsilon$ и шаг $\alpha$ — они понадобятся
на жёлтом и красном уровнях.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, LogNorm
from matplotlib.patches import Patch

plt.rcParams["figure.dpi"] = 110

## 1. Классификация исходов и допуски

Для каждой начальной точки возможен один из исходов:

| код | исход | условие |
|---|---|---|
| $k = 0, 1, 2$ | сошлась к корню $r_k$ | $\lvert z_n - r_k\rvert < \delta_z$ и $\lvert f(z_n)\rvert < \delta_f$ |
| $-2$ | численный сбой | $\lvert f'(z_n)\rvert < \delta_{f'}$ или $z_n$ перестало быть конечным числом |
| $-1$ | не определено | за $N_{\max}$ итераций не выполнилось ни одно из условий выше |

Допуски: $\delta_z = 10^{-8}$, $\delta_f = 10^{-10}$, $\delta_{f'} = 10^{-14}$. Расстояние
между корнями при $\varepsilon = 0$ равно 1, так что $\delta_z$ на много порядков меньше
него, и спутать корни нельзя. У простого корня $|f(z)| \approx |f'(r)|\,|z - r|$, поэтому
одно условие на невязку здесь почти равносильно условию на расстояние. Оба условия всё равно
проверяю вместе: при сближении корней (жёлтый уровень) малая невязка уже не гарантирует
близость к конкретному корню.

Порядок действий на итерации: сначала проверяю, не сошлась ли точка, и только потом делю на
производную. Иначе точка, уже сидящая в корне, может попасть в «сбой» из-за маленькой $f'$ —
у двойного корня при $\varepsilon = 1$ производная в самом корне равна нулю.

Активные точки хранятся в сжатом массиве индексов: сошедшиеся и сбойные точки выбывают и
больше не пересчитываются.

In [ ]:
UNDETERMINED, FAILURE = -1, -2
TOL_Z, TOL_F, TOL_D = 1e-8, 1e-10, 1e-14


def roots_of(eps):
    return np.array([-1.0, 1.0, eps], dtype=complex)


def f_and_df(z, roots):
    d = z[..., None] - roots
    f = np.prod(d, axis=-1)
    df = (d[..., 1] * d[..., 2] + d[..., 0] * d[..., 2] + d[..., 0] * d[..., 1])
    return f, df


def newton_classify(Z0, eps=0.0, alpha=1.0, n_max=100, tol_z=TOL_Z, tol_f=TOL_F, tol_d=TOL_D):
    roots = roots_of(eps)
    shape = np.shape(Z0)
    z = np.array(Z0, dtype=complex).ravel().copy()
    code = np.full(z.size, UNDETERMINED, dtype=int)
    n_iter = np.full(z.size, n_max, dtype=int)
    idx = np.arange(z.size)

    for n in range(n_max + 1):
        zi = z[idx]
        f, df = f_and_df(zi, roots)

        dist = np.abs(zi[:, None] - roots[None, :])
        k = np.argmin(dist, axis=1)
        conv = (dist[np.arange(len(idx)), k] < tol_z) & (np.abs(f) < tol_f)
        code[idx[conv]] = k[conv]
        n_iter[idx[conv]] = n

        bad = ~conv & ((np.abs(df) < tol_d) | ~np.isfinite(zi))
        code[idx[bad]] = FAILURE
        n_iter[idx[bad]] = n

        keep = ~(conv | bad)
        if n == n_max or not keep.any():
            break
        idx, zi, f, df = idx[keep], zi[keep], f[keep], df[keep]
        z[idx] = zi - alpha * f / df

    return code.reshape(shape), n_iter.reshape(shape), z.reshape(shape)


def newton_path(z0, eps=0.0, alpha=1.0, n_max=100):
    roots = roots_of(eps)
    path, z = [complex(z0)], complex(z0)
    for _ in range(n_max):
        f, df = f_and_df(np.array([z]), roots)
        f, df = f[0], df[0]
        if np.min(np.abs(z - roots)) < TOL_Z and abs(f) < TOL_F:
            break
        if abs(df) < TOL_D or not np.isfinite(z):
            break
        z = z - alpha * f / df
        path.append(z)
    return np.array(path)

## 2. Бассейны при $\varepsilon = 0$, $N_{\max} = 100$

In [ ]:
N, EXT = 600, 2.0
x = np.linspace(-EXT, EXT, N)
Z0 = x[None, :] + 1j * x[:, None]
extent = (-EXT, EXT, -EXT, EXT)
eps = 0.0
roots = roots_of(eps)

code100, nit100, _ = newton_classify(Z0, eps=eps, n_max=100)

ROOT_COLORS = ["#4c72b0", "#dd8452", "#55a868"]
CAT_COLORS = {UNDETERMINED: "#000000", FAILURE: "#ff00ff"}


def category_image(code):
    img = np.empty(code.shape, dtype=int)
    for k in range(3):
        img[code == k] = k
    img[code == UNDETERMINED] = 3
    img[code == FAILURE] = 4
    return img


def show_basins(ax, code, n_iter, title, shade=True):
    cmap = ListedColormap(ROOT_COLORS + [CAT_COLORS[UNDETERMINED], CAT_COLORS[FAILURE]])
    ax.imshow(category_image(code), extent=extent, origin="lower", cmap=cmap,
              vmin=-0.5, vmax=4.5, interpolation="nearest")
    if shade:
        conv = code >= 0
        shade_img = np.where(conv, np.clip(np.log(n_iter + 1) / np.log(40), 0, 1), np.nan)
        ax.imshow(shade_img, extent=extent, origin="lower", cmap="Greys", alpha=0.45,
                  vmin=0, vmax=1, interpolation="nearest")
    for k, r in enumerate(roots):
        ax.plot(r.real, r.imag, "o", ms=9, mfc="white", mec="black", mew=1.5)
    ax.set_aspect("equal")
    ax.set_xlabel(r"$\mathrm{Re}\,z_0$"); ax.set_ylabel(r"$\mathrm{Im}\,z_0$")
    ax.set_title(title, fontsize=10)


def legend_handles(code):
    h = [Patch(color=ROOT_COLORS[k], label=f"корень $r_{k}$ = {roots[k].real:g} ({np.sum(code == k)} точек)")
         for k in range(3)]
    h.append(Patch(color=CAT_COLORS[UNDETERMINED], label=f"не определено за $N_{{max}}$ ({np.sum(code == UNDETERMINED)})"))
    h.append(Patch(color=CAT_COLORS[FAILURE], label=f"численный сбой, $|f'| < \\delta_{{f'}}$ ({np.sum(code == FAILURE)})"))
    h.append(plt.Line2D([], [], ls="", marker="o", mfc="white", mec="black", label="корни"))
    return h


fig, ax = plt.subplots(figsize=(8.5, 7.5))
show_basins(ax, code100, nit100,
            f"Бассейны Ньютона, $\\varepsilon$ = 0: сетка {N}×{N}, $N_{{max}}$ = 100\n"
            f"$\\delta_z$ = {TOL_Z:g}, $\\delta_f$ = {TOL_F:g}, $\\delta_{{f'}}$ = {TOL_D:g}; "
            "затемнение — больше итераций")
ax.legend(handles=legend_handles(code100), loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8)
fig.tight_layout()
plt.show()

Критические точки $f$ здесь — $z = \pm 1/\sqrt3 \approx \pm0.577$: в них касательная
горизонтальна, и шаг Ньютона уходит на бесконечность. Вокруг них сходятся «лепестки» всех трёх
бассейнов. На сетке ни одна точка в них точно не попадает, поэтому категория «сбой» на карте
пустая; отдельно она проверена ниже на траекториях.

## 3. Число итераций до сходимости

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 7.5))
cmap_it = plt.get_cmap("viridis").copy()
cmap_it.set_bad("black")
data = np.ma.masked_where(code100 < 0, nit100 + 1)
im = ax.imshow(data, extent=extent, origin="lower", cmap=cmap_it,
               norm=LogNorm(vmin=1, vmax=101), interpolation="nearest")
for r in roots:
    ax.plot(r.real, r.imag, "o", ms=9, mfc="white", mec="black", mew=1.5)
ax.set_aspect("equal")
ax.set_xlabel(r"$\mathrm{Re}\,z_0$"); ax.set_ylabel(r"$\mathrm{Im}\,z_0$")
ax.set_title("Число итераций Ньютона до сходимости, $N_{max}$ = 100\nчёрный — не определено или сбой")
fig.colorbar(im, ax=ax, label="число итераций + 1 (лог. шкала)", shrink=0.85)
fig.tight_layout()
plt.show()

conv = code100 >= 0
print(f"сошлось: {conv.mean() * 100:.3f}% точек; медиана числа итераций {np.median(nit100[conv]):.0f}, "
      f"максимум {nit100[conv].max()}")
for q in [50, 90, 99, 99.9]:
    print(f"  {q}% сошедшихся точек укладываются в {np.percentile(nit100[conv], q):.0f} итераций")

Половине точек хватает 6 итераций, 99% — 13, самой медленной точке сетки — 24: около
простого корня Ньютон сходится квадратично. Число
итераций растёт к границам бассейнов — там точка долго мечется между лепестками, прежде чем
попасть в окрестность одного из корней.

## 4. Повтор с увеличенным лимитом итераций

При $N_{\max} = 100$ сходятся все точки сетки (максимум 24 итерации), так что неопределённых точек
нет. Чтобы увидеть, где они появляются и куда деваются, считаю ещё и с заведомо малыми лимитами:
$N_{\max} = 5, 10, 20$, а также с увеличенным $N_{\max} = 1000$.

In [ ]:
runs = {nm: newton_classify(Z0, eps=eps, n_max=nm)[:2] for nm in [5, 10, 20, 100, 1000]}
code1000 = runs[1000][0]

labels = {0: "r0 = -1", 1: "r1 = 1", 2: "r2 = 0", UNDETERMINED: "не определено", FAILURE: "сбой"}
print(f"{'исход':>15}" + "".join(f"{'N_max=' + str(nm):>12}" for nm in runs))
for c in [0, 1, 2, UNDETERMINED, FAILURE]:
    print(f"{labels[c]:>15}" + "".join(f"{np.sum(runs[nm][0] == c):12d}" for nm in runs))

print()
for nm in [5, 10, 20, 100]:
    c_small = runs[nm][0]
    conv = c_small >= 0
    switched = np.sum(c_small[conv] != code1000[conv])
    undet = c_small == UNDETERMINED
    print(f"N_max = {nm:>3}: сошедшихся точек, сменивших корень при N_max = 1000: {switched}; "
          f"неопределённых {undet.sum()}, из них при N_max = 1000 сошлись {np.sum(code1000[undet] >= 0)}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5.8))
for ax, nm in zip(axes, [5, 10, 100]):
    c, nit = runs[nm]
    show_basins(ax, c, nit, f"$N_{{max}}$ = {nm}: не определено — {np.sum(c == UNDETERMINED)} точек")
    ax.plot([1 / np.sqrt(3), -1 / np.sqrt(3)], [0, 0], "x", color="magenta", ms=10, mew=2)
axes[0].legend(handles=legend_handles(runs[5][0]), loc="upper left", fontsize=7, framealpha=0.9)
fig.tight_layout()
plt.show()

При $N_{\max} = 5$ не определены даже точки далеко от корней: им просто не хватает шагов, чтобы
дойти до окрестности корня. При $N_{\max} = 10$ чёрными остаются только полосы вдоль границ
бассейнов и окрестности критических точек $z = \pm1/\sqrt3$ (крестики), при $N_{\max} = 20$ —
68 точек у самых границ. Все неопределённые точки при большем лимите досчитываются и попадают в
тот же бассейн, что и при $N_{\max} = 1000$; ни одна точка, сошедшаяся при малом лимите, корень
не меняет.

## 5. Последовательности приближений из разных начальных точек

Выбрал точки с разными исходами:

| $z_0$ | что ожидается |
|---|---|
| $1.5 + 1i$ | обычная сходимость к $r_1 = 1$ |
| $-0.3 + 1.2i$ | сходимость к корню после нескольких «прыжков» |
| $0.55$ | вещественная точка между бассейнами: улетает далеко и возвращается |
| $1/\sqrt5$ | в точной арифметике — 2-цикл $1/\sqrt5 \leftrightarrow -1/\sqrt5$ |
| $1/\sqrt3$ | $f'(z_0) = 0$, численный сбой на первом шаге |

In [ ]:
starts = [1.5 + 1.0j, -0.3 + 1.2j, 0.55 + 0j, 1 / np.sqrt(5) + 0j, 1 / np.sqrt(3) + 0j]
names = ["1.5+1i", "-0.3+1.2i", "0.55", "1/√5", "1/√3"]
outcome_code, outcome_n, _ = newton_classify(np.array(starts), eps=eps, n_max=100)

print(f"{'z0':>10} {'исход':>15} {'шагов':>6} {'|f(z0)|':>10} {'|f\'(z0)|':>10}")
for z0, nm, c, n in zip(starts, names, outcome_code, outcome_n):
    f0, df0 = f_and_df(np.array([z0]), roots)
    print(f"{nm:>10} {labels[c]:>15} {n:6d} {abs(f0[0]):10.2e} {abs(df0[0]):10.2e}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6.2))
show_basins(axes[0], code100, nit100, "Траектории итераций на фоне бассейнов", shade=False)
for z0, nm, col in zip(starts, names, ["k", "w", "red", "yellow", "magenta"]):
    p = newton_path(z0, eps=eps)
    vis = np.abs(p) < 2
    axes[0].plot(p.real, p.imag, "-", color=col, lw=1, alpha=0.9)
    axes[0].plot(p.real[vis], p.imag[vis], "o", color=col, ms=3)
    axes[0].plot(p.real[0], p.imag[0], "*", color=col, ms=13, mec="black", label=nm)
axes[0].set_xlim(-2, 2); axes[0].set_ylim(-2, 2)
axes[0].annotate("из 0.55 первый шаг\nуводит в Re z ≈ −3.6", (-2, 0), xytext=(-1.95, -0.6),
                 color="red", fontsize=8, arrowprops=dict(arrowstyle="->", color="red"))
axes[0].legend(loc="upper left", fontsize=8, framealpha=0.9)

for z0, nm, col in zip(starts[:4], names[:4], ["C0", "C1", "C3", "C8"]):
    p = newton_path(z0, eps=eps)
    err = np.min(np.abs(p[:, None] - roots[None, :]), axis=1)
    axes[1].semilogy(np.arange(len(p)), err, "o-", ms=3, color=col, label=f"$z_0$ = {nm}")
axes[1].axhline(TOL_Z, color="0.5", ls="--", lw=0.8)
axes[1].text(0.3, TOL_Z * 2, r"$\delta_z$", color="0.4")
axes[1].set_xlabel("номер итерации $n$")
axes[1].set_ylabel(r"$\min_k |z_n - r_k|$")
axes[1].set_title("Расстояние до ближайшего корня")
axes[1].legend(fontsize=8)
fig.tight_layout()
plt.show()

Для $z_0 = 1/\sqrt5$: $N(z) = 2z^3/(3z^2 - 1)$, и $N(1/\sqrt5) = -1/\sqrt5$: в точной
арифметике точка вечно прыгает по 2-циклу. Цикл отталкивающий: $N'(z) = f f''/f'^2 = -6$ в
каждой из двух точек, мультипликатор цикла 36. Ошибка округления $\sim10^{-16}$ растёт в 36 раз
за цикл, и примерно через десяток циклов траектория уходит с цикла и сходится к корню — это
видно по горизонтальному участку на правом графике.

## 6. Ответ на вопрос

**Можно ли считать точку вне бассейнов корней только потому, что итерации не сошлись за
отведённое число шагов?**

Нет. «Не определено» означает лишь, что за $N_{\max}$ шагов точка не подошла ни к одному корню
ближе $\delta_z$. Причин может быть три, и по одному счётчику итераций их не различить.

1. Медленная сходимость. Около границы бассейнов точка долго блуждает, прежде чем попасть в
   окрестность корня. Это видно по таблице в разделе 4: при $N_{\max} = 10$ не определены
   18620 точек, при $N_{\max} = 20$ — 68, при $N_{\max} = 100$ — ни одной, и все они
   сходятся к тому же корню, что и при $N_{\max} = 1000$. Если бы я объявил их «вне бассейнов»
   при $N_{\max} = 10$, ошибся бы для всех 18620 точек.
2. Точка лежит на самой границе бассейнов — на множестве Жюлиа отображения Ньютона: прообразы
   критических точек $\pm1/\sqrt3$, отталкивающие циклы вроде $\pm1/\sqrt5$. Такие точки
   действительно никуда не сходятся. Но при $\varepsilon = 0$ все критические точки $N$
   лежат в бассейнах корней (см. п. 3), отображение гиперболическое, и множество Жюлиа имеет
   нулевую площадь: на сетку такие точки почти не попадают, а в плавающей арифметике округление само выталкивает траекторию с
   отталкивающего цикла.
3. Притягивающий цикл, отличный от корней. Тогда целая открытая область действительно лежит вне
   бассейнов. Но для $\varepsilon = 0$ это исключено строго: по теореме Фату каждый
   притягивающий цикл притягивает хотя бы одну критическую точку отображения
   $N(z) = z - f/f'$. Критические точки $N$ — это корни $f$ (где $N' = 0$) и нули $f''$; здесь
   $f'' = 6z$ обращается в ноль только в корне $z = 0$. Свободных критических точек нет, значит,
   других притягивающих циклов тоже нет, и почти все точки плоскости сходятся к корням.

Вывод: точку можно считать лежащей вне бассейнов, только если это доказано — например, найден
притягивающий цикл, к которому она сходится. Неудача за конечное число шагов означает лишь
«исход не установлен». Для $\varepsilon \ne 0$ у $f_\varepsilon$ появляется свободная критическая
точка $N$ — нуль $f''$ в $z = \varepsilon/3$, не совпадающий с корнями, — и вопрос о
притягивающих циклах нужно проверять заново. Это задел для жёлтого уровня.